# SSA vs ODE

Each experiment runs the **stochastic SSA** (Gillespie, ensemble-averaged moments)
and the **deterministic ODE** moment equations for the *same* parameters, then plots
both side by side.

**Goal:** check that the ODE = exact moment dynamics, and the SSA estimates converge
to it.

In [1]:
import sys
from pathlib import Path


sys.path.append(str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2

import numpy as np

from src.ssa_simulation import simulate_telegraph, compute_sample_moments
from src.ode_moments import solve_ode_moments
from src.comparison_visualization import show_combined_moments

In [2]:
def run_comparison(title: str, n_grid=1000, time_step=None, t_end=None, **kwargs):
    base_params = {
        "k_on": 0.1,
        "k_off": 0.1,
        "k_syn": 10.0,
        "k_deg": 1,
        "t0": 0.0,
        "g0": 0,
        "r0": 0,
        "n_sim": 4000,
        "n_rep": 1000,
    }

    current_params = {**base_params, **kwargs}

    data = simulate_telegraph(**current_params)

    # One shared window for SSA and ODE. Default: earliest-finishing trajectory,
    # so every cell fully covers it (no held tails). Pass t_end explicitly to pin
    # a fixed window; trajectories that ended earlier then hold their last value (ZOH).
    if t_end is None:
        t_end = float(data[-1, :, 0].min())

    moments_ssa = compute_sample_moments(
        data, t_end=t_end, n_grid=n_grid, time_step=time_step
    )

    t_ode, y_ode = solve_ode_moments(
        k_on=current_params["k_on"],
        k_off=current_params["k_off"],
        k_syn=current_params["k_syn"],
        k_deg=current_params["k_deg"],
        t0=current_params["t0"],
        g0=current_params["g0"],
        r0=current_params["r0"],
        t_end=t_end,
    )

    show_combined_moments(moments_ssa, t_ode, y_ode, title=title)

### 1. Switching speed — **1a. Slow** ($k_{\text{on}}=0.05,\; k_{\text{off}}=0.05 \ll k_{\text{deg}}=1$)

Gene flips rarely → long ON/OFF bursts → large variance. (`k_syn=20`, ⟨R⟩≈10.)

In [3]:
run_comparison(
    title="Experiment 1a: Slow Gene Switching Regime",
    k_on=0.05,
    k_off=0.05,
    k_syn=20.0,
)

**1b. Fast** ($k_{\text{on}}=10,\; k_{\text{off}}=10 \gg k_{\text{deg}}=1$)

Rapid flickering, promoter effectively averaged → low variance. (Same ⟨R⟩≈10 as 1a.)

In [4]:
run_comparison(
    title="Experiment 1b: Fast Gene Switching Regime",
    k_on=10.0,
    k_off=10.0,
    k_syn=20.0,
)

### 2. Expression level — **2a. High** ($k_{\text{on}}=0.5,\; k_{\text{off}}=4,\; k_{\text{syn}}=56$)

TATA + Inr: large burst size ($k_{\text{syn}}/k_{\text{off}}\approx 14$) → high steady-state ⟨R⟩.

In [5]:
run_comparison(
    title="Experiment 2a: High Expression Regime",
    k_on=0.5,
    k_off=4,
    k_syn=56,
)

**2b. Low** ($k_{\text{on}}=0.5,\; k_{\text{off}}=10,\; k_{\text{syn}}=30$)

TATA-less / Inr-less: small burst size ($k_{\text{syn}}/k_{\text{off}}\approx 3$) → few molecules;
the SSA mean should still match the ODE.

In [6]:
run_comparison(
    title="Experiment 2b: Low Expression Regime",
    k_on=0.5,
    k_off=10,
    k_syn=30,
)

### 3. Initial conditions — **3a. Inactive start** ($g_0=0,\; r_0=0$)

Gene resting, no mRNA → ⟨R⟩ builds up from zero.

In [7]:
run_comparison(
    title="Experiment 3a: Inactive Start (g0=0, r0=0)",
    g0=0,
    r0=0,
)

**3b. Active-burst start** ($g_0=1,\; r_0=15$)

⟨R⟩ relaxes down to the same steady state — ODE gives the exact relaxation curve.

In [8]:
run_comparison(
    title="Experiment 3b: Active-Burst Start (g0=1, r0=15)",
    g0=1,
    r0=15,
)

### 4. Numerical parameters — **4a. Low scale** ($n_{\text{sim}}=1000$)

Short SSA runs capture only a few bursts; the ODE is the noise-free reference.

In [9]:
run_comparison(
    title="Experiment 4a: Low Scale (n_sim=1000)",
    n_sim=1000,
)

**4b. Experiment scale** ($n_{\text{rep}}=224,\; n_{\text{sim}}=5000$)

$n_{\text{rep}}$ matches the 224 sequenced fibroblasts; long runs reach steady state, where SSA
converges to the ODE.

In [10]:
run_comparison(
    title="Experiment 4b: Experiment Scale (n_rep=224, n_sim=5000)",
    n_rep=224,
    n_sim=5000,
)